### 1. Setup e Imports

In [ ]:
# Instalar SAELens si no está disponible:
# !pip install sae-lens

import json as _json
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import os
import sys
from pathlib import Path

# Agregar path del proyecto para imports
sys.path.append(os.path.abspath('../../..'))
from sae.tools.naming_utils import save_checkpoint, get_model_name, get_extras_id, get_checkpoint_dir
from sae.tools.experiment_utils import get_experiment_dir, get_plots_dir, get_metrics_dir, get_report_name

# Configuración de visualización para Quarto/PDF
pd.set_option('display.width', 100)
pd.set_option('display.max_columns', None)
np.set_printoptions(linewidth=100, precision=4, suppress=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {device}')


### 2 Metadata para Naming de Checkpoints

In [ ]:
# Metadata para naming de checkpoints (separada de hiperparámetros)
NAMING_META = {
    'variante': 'topk',              # Arquitectura del SAE
    'tecnica': 'topk',               # TopK: sparsity por hard constraint (k features)
    'capa': 6,                       # Layer del modelo OthelloGPT
    'juegos': 200,                   # Número de partidas
    'base_path': 'C:\\Users\\Esposa\\Documents\\Repos\\sae-othello-gpt',
    'experiment_base_path': os.path.abspath('../../../experiments'),
    'extras': {
        'expansion': 16,   # d_sae / d_in = 8192 / 512
        'k': 50,           # número de features activas (L0 fijo)
        'epochs': 30,
        'patience': 5
    }
}

print('\nMetadata de naming:')
for key, value in NAMING_META.items():
    print(f'  {key}: {value}')


In [ ]:
extras_id = get_extras_id(
    get_checkpoint_dir(
        NAMING_META['base_path'],
        NAMING_META['variante'],
        NAMING_META['tecnica'],
        NAMING_META['capa'],
        NAMING_META['juegos']
    ),
    NAMING_META['extras']
) if NAMING_META.get('extras') else None

# get_plots_dir crea el directorio automáticamente
plots_dir = Path(get_plots_dir(
    NAMING_META['experiment_base_path'],
    NAMING_META['variante'],
    NAMING_META['tecnica'],
    NAMING_META['capa'],
    NAMING_META['juegos'],
    extras_id=extras_id
))

print(f'extras_id: {extras_id}')
print(f'plots_dir: {plots_dir}')

### 3. Configuración del SAE

Configuración de hiperparámetros en un diccionario `cfg` con los mismos 
nombres que usa SAELens, para facilitar comparación futura.

**Arquitectura TopK:**
- **TopK activation**: mantiene exactamente `k` features activas por muestra (L0 fijo = k)
- **Sin coeficiente de sparsidad**: la sparsity se controla directamente con `k`
- **Loss = MSE únicamente** (sin penalización L1)
...

In [ ]:
NUM_EPOCHS = NAMING_META['extras']['epochs']

cfg = {
    'architecture':          'topk',
    'd_in':                  512,
    'd_sae':                 8192,
    'k':                     50,
    'lr':                    1e-3,
    'train_batch_size_tokens': 128,
    'hook_name':             'blocks.5.hook_resid_post', #ojo
}

cfg_summary = pd.DataFrame({
    'Parámetro SAELens': ['architecture', 'd_in', 'd_sae', 'k (L0 fijo)', 'hook_name', 'lr'],
    'Valor': [cfg['architecture'], cfg['d_in'], cfg['d_sae'], cfg['k'],
              cfg['hook_name'], cfg['lr']]
})
print('Configuración SAELens:')
print(cfg_summary.to_string(index=False))

### 4. Dataset de Activaciones (Train/Validation Split 80-20%)

Activaciones pre-extraídas con `sae/activations/run_extraction.py` desde la capa `blocks.5.hook_resid_post`.


In [ ]:
class ActivationsDataset(Dataset):
    """Dataset para cargar activaciones extraídas"""
    def __init__(self, activations_path):
        self.activations = np.load(activations_path)
    def __len__(self):
        return len(self.activations)
    def __getitem__(self, idx):
        return torch.tensor(self.activations[idx], dtype=torch.float32)

activations_file = Path(f"../../../activations/data/layer{NAMING_META['capa']}_{NAMING_META['juegos']}games.npy")
full_dataset = ActivationsDataset(activations_file)

# Split 80-20 reproducible
train_size = int(0.8 * len(full_dataset))
val_size   = len(full_dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(
    full_dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_dataset, batch_size=cfg["train_batch_size_tokens"], shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=cfg["train_batch_size_tokens"], shuffle=False)

dataset_info = pd.DataFrame({
    'Conjunto': ['Train', 'Validación', 'Total'],
    'Muestras': [f'{len(train_dataset):,}', f'{len(val_dataset):,}', f'{len(full_dataset):,}'],
    'Shape':    [str(full_dataset.activations.shape), '', ''],
})
print(dataset_info.to_string(index=False))


### 5. Modelo: TopK SAE

Implementación de la arquitectura TopK de SAELens.

**Diferencia clave vs ReLU + L1:**
- El encoder proyecta a `d_sae=8192` dimensiones (pre-activación)
- La activación **TopK** retiene solo los `k=50` valores más altos; el resto = 0
- L0 es **exactamente `k`** para cada muestra (hard constraint, no soft penalty)
- No hay `sparsity_coef` ni cosine annealing
- El decoder se normaliza a norma unitaria después de cada paso del optimizador


In [ ]:
class TopKSAE(nn.Module):
    """
    Sparse Autoencoder con activación TopK.
    Equivalente a SAELens architecture='topk'.

    Encoder    : d_in -> d_sae  (lineal + pre-bias centering) #que es prebias centering
    Activación : TopK  (mantiene exactamente k features activas, clamp(min=0))
    Decoder    : d_sae -> d_in  (columnas con norma unitaria)
    """
    def __init__(self, d_in: int, d_sae: int, k: int):
        super().__init__()
        self.d_in  = d_in
        self.d_sae = d_sae
        self.k     = k

        self.W_enc = nn.Parameter(torch.empty(d_in, d_sae))
        self.b_enc = nn.Parameter(torch.zeros(d_sae))
        self.W_dec = nn.Parameter(torch.empty(d_sae, d_in))
        self.b_dec = nn.Parameter(torch.zeros(d_in))

        nn.init.kaiming_uniform_(self.W_enc)
        nn.init.kaiming_uniform_(self.W_dec)
        self.normalize_decoder()

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        """Proyecta a espacio latente con activación TopK"""
        x_cent   = x - self.b_dec          # centrado en torno al decoder bias (como SAELens)
        pre_acts = x_cent @ self.W_enc + self.b_enc
        return self._topk_activation(pre_acts)

    def _topk_activation(self, x: torch.Tensor) -> torch.Tensor:
        """Mantiene exactamente k activaciones por muestra, resto -> 0"""
        topk_vals, topk_idx = torch.topk(x, self.k, dim=-1)
        result = torch.zeros_like(x)
        result.scatter_(-1, topk_idx, topk_vals)
        return result.clamp(min=0)         # solo activaciones positivas

    def decode(self, hidden: torch.Tensor) -> torch.Tensor:
        return hidden @ self.W_dec + self.b_dec

    def forward(self, x: torch.Tensor):
        hidden = self.encode(x)
        recon  = self.decode(hidden)
        return recon, hidden

    @torch.no_grad()
    def normalize_decoder(self):
        """Normaliza filas de W_dec a norma unitaria (columnas del diccionario)"""
        norms = self.W_dec.data.norm(dim=1, keepdim=True)
        self.W_dec.data.div_(norms.clamp(min=1e-8))


# Crear modelo desde parámetros del cfg SAELens
model = TopKSAE(d_in=cfg['d_in'], d_sae=cfg['d_sae'], k=cfg['k']).to(device)
num_params = sum(p.numel() for p in model.parameters())

model_info = pd.DataFrame({
    'Componente': [
        'Input', 'Encoder (W_enc)', 'Activación', 'Hidden activos', 'Decoder (W_dec)', 'Output', 'Total params'
    ],
    'Dimensión': [
        f'{cfg["d_in"]}',
        f'{cfg["d_in"]} -> {cfg["d_sae"]}',
        f'TopK (k={cfg["k"]})',
        f'{cfg["d_sae"]} (solo {cfg["k"]} activos)',
        f'{cfg["d_sae"]} -> {cfg["d_in"]}',
        f'{cfg["d_in"]}',
        f'{num_params:,}'
    ]
})
print(model_info.to_string(index=False))


### 6. Función de Pérdida TopK

**Loss = MSE(reconstrucción)**

- **MSE**: error de reconstrucción (igual que en L1)
- **Sin L1 penalty**: la sparsity ya está impuesta de forma exacta por la activación TopK
- No hay `sparsity_coef` ni annealing — el único hiperparámetro de sparsity es `k`


In [ ]:
def loss_function(x, reconstruction, hidden):
    """
    Pérdida TopK: solo MSE.
    La sparsity ya la impone la activación TopK (L0 = k exacto).

    Args:
        x:              Activaciones originales
        reconstruction: Activaciones reconstruidas
        hidden:         Activaciones latentes (exactamente k activas)
    Returns:
        mse_loss (scalar)
    """
    return F.mse_loss(reconstruction, x)


optimizer = torch.optim.Adam(model.parameters(), lr=cfg["lr"])

optim_info = pd.DataFrame({
    'Parámetro': ['Optimizador', 'Learning rate', 'k (L0 fijo)', 'Sparsity coef'],
    'Valor':     ['Adam', cfg["lr"], cfg["k"], 'N/A — TopK no necesita penalización']
})
print(optim_info.to_string(index=False))


### 7. Loop de Entrenamiento

In [ ]:
PATIENCE  = NAMING_META['extras']['patience']

save_dir = Path('./saved_model')
save_dir.mkdir(exist_ok=True)
best_model_name = get_model_name(
    variante=NAMING_META['variante'],
    tecnica=NAMING_META['tecnica'],
    capa=NAMING_META['capa'],
    juegos=NAMING_META['juegos'],
    estado='best'
)
BEST_CKPT = save_dir / best_model_name

history = {
    'train_mse_loss':        [],
    'train_l0_sparsity':     [],
    'train_fraction_active': [],
    'val_mse_loss':          [],
    'val_l0_sparsity':       [],
    'val_fraction_active':   [],
}

best_val_mse      = float('inf')
best_epoch        = 0
epochs_no_improve = 0

print(f'Iniciando entrenamiento: {NUM_EPOCHS} épocas máximo')
print(f'Early stopping: patience={PATIENCE}')
print(f'k = {cfg["k"]} features activas (L0 fijo)')
print(f'Best checkpoint: {BEST_CKPT}')
print('=' * 70)

for epoch in range(NUM_EPOCHS):

    # TRAIN
    model.train()
    t_mse = t_l0 = t_frac = 0.0

    for batch in train_loader:
        x = batch.to(device)
        optimizer.zero_grad()
        recon, hidden = model(x)
        mse = loss_function(x, recon, hidden)
        mse.backward()
        optimizer.step()
        model.normalize_decoder()  # norma unitaria en columnas del decoder

        with torch.no_grad():
            t_mse  += mse.item()
            t_l0   += (hidden > 0).float().sum(dim=1).mean().item()
            t_frac += (hidden > 0).float().mean().item()

    n = len(train_loader)
    history['train_mse_loss'].append(t_mse / n)
    history['train_l0_sparsity'].append(t_l0 / n)
    history['train_fraction_active'].append(t_frac / n)

    # VALIDATION
    model.eval()
    v_mse = v_l0 = v_frac = 0.0

    with torch.no_grad():
        for batch in val_loader:
            x = batch.to(device)
            recon, hidden = model(x)
            v_mse  += loss_function(x, recon, hidden).item()
            v_l0   += (hidden > 0).float().sum(dim=1).mean().item()
            v_frac += (hidden > 0).float().mean().item()

    nv = len(val_loader)
    history['val_mse_loss'].append(v_mse / nv)
    history['val_l0_sparsity'].append(v_l0 / nv)
    history['val_fraction_active'].append(v_frac / nv)

    print(f"Época {epoch+1:3d}/{NUM_EPOCHS} | "
          f"Train MSE: {history['train_mse_loss'][-1]:.6f} | "
          f"Val MSE: {history['val_mse_loss'][-1]:.6f} | "
          f"L0: {history['val_l0_sparsity'][-1]:.1f}")

    # Early stopping
    if history['val_mse_loss'][-1] < best_val_mse:
        best_val_mse      = history['val_mse_loss'][-1]
        best_epoch        = epoch + 1
        epochs_no_improve = 0
        torch.save(model.state_dict(), BEST_CKPT)
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f'\nEarly stopping en época {epoch+1} (mejor: época {best_epoch})')
            break

# Restaurar mejor modelo
model.load_state_dict(torch.load(BEST_CKPT, weights_only=True))
print(f'\nMejor época: {best_epoch} | Mejor Val MSE: {best_val_mse:.6f}')

### 8. Guardar Modelo

In [ ]:
# Guardar checkpoint final
checkpoint = {
    'model_state_dict':     model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'config': {
        'architecture': cfg["architecture"],
        'd_in':         cfg["d_in"],
        'd_sae':        cfg["d_sae"],
        'k':            cfg["k"],
        'lr':           cfg["lr"],
        'hook_name':    cfg["hook_name"],
    },
    'history': history,
}

final_model_name = get_model_name(
    variante=NAMING_META['variante'],
    tecnica=NAMING_META['tecnica'],
    capa=NAMING_META['capa'],
    juegos=NAMING_META['juegos'],
    estado='final'
)
checkpoint_path = save_dir / final_model_name
torch.save(checkpoint, checkpoint_path)
print(f'Modelo guardado: {checkpoint_path}')

save_checkpoint(
    model,
    base_path=NAMING_META['base_path'],
    variante=NAMING_META['variante'],
    tecnica=NAMING_META['tecnica'],
    capa=NAMING_META['capa'],
    juegos=NAMING_META['juegos'],
    estado='final',
    extras=NAMING_META['extras'],
    extras_id=extras_id
)

### 9. Métricas básicas del modelo

In [ ]:
import json

metrics_dir = get_metrics_dir(
    NAMING_META['experiment_base_path'],
    NAMING_META['variante'],
    NAMING_META['tecnica'],
    NAMING_META['capa'],
    NAMING_META['juegos'],
    extras_id=extras_id
)

training_metrics = {
    # Calidad de reconstrucción
    'val_mse_final':             history['val_mse_loss'][-1],
    'val_mse_best':              best_val_mse,
    # Sparsity (L0 = k siempre para TopK)
    'val_l0_final':              history['val_l0_sparsity'][-1],
    'val_fraction_active_final': history['val_fraction_active'][-1],
    # Contexto del entrenamiento
    'best_epoch':                best_epoch,
    'total_epochs':              len(history['train_mse_loss']),
    'train_mse_final':           history['train_mse_loss'][-1],
    'train_l0_final':            history['train_l0_sparsity'][-1],
    # Hiperparámetros TopK
    'architecture':              cfg["architecture"],
    'k':                         cfg["k"],
    'd_sae':                     cfg["d_sae"],
    'd_in':                      cfg["d_in"],
    'expansion_factor':          cfg["d_sae"] // cfg["d_in"],
    'hook_name':                 cfg["hook_name"],
}

os.makedirs(metrics_dir, exist_ok=True)
metrics_path = Path(metrics_dir) / 'training_metrics.json'
with open(metrics_path, 'w') as f:
    json.dump(training_metrics, f, indent=2)

print(f'Métricas guardadas: {metrics_path}')
print(pd.DataFrame(
    list(training_metrics.items()), columns=['Métrica', 'Valor']
).to_string(index=False))


### 10. Visualización de Resultados

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# MSE Loss
axes[0].plot(history['train_mse_loss'], label='Train', color='steelblue', alpha=0.8)
axes[0].plot(history['val_mse_loss'],   label='Validación', color='tomato', alpha=0.8)
if best_epoch > 0:
    axes[0].axvline(best_epoch - 1, color='green', linestyle='--', alpha=0.7,
                    label=f'Best (época {best_epoch})')
axes[0].set_title('MSE Loss (Reconstrucción)')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('MSE')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# L0 sparsity (para TopK es constante = k)
axes[1].plot(history['train_l0_sparsity'], label='Train', color='steelblue', alpha=0.8)
axes[1].plot(history['val_l0_sparsity'],   label='Validación', color='tomato', alpha=0.8)
axes[1].axhline(cfg["k"], color='orange', linestyle='--', alpha=0.8, label=f'k={cfg["k"]} (target)')
axes[1].set_title(f'L0 Sparsity (esperado = k={cfg["k"]})')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Features activas')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Fracción activa
axes[2].plot(history['train_fraction_active'], label='Train', color='steelblue', alpha=0.8)
axes[2].plot(history['val_fraction_active'],   label='Validación', color='tomato', alpha=0.8)
expected_frac = cfg["k"] / cfg["d_sae"]
axes[2].axhline(expected_frac, color='orange', linestyle='--', alpha=0.8,
                label=f'k/d_sae={expected_frac:.4f}')
axes[2].set_title('Fracción Features Activas')
axes[2].set_xlabel('Época')
axes[2].set_ylabel('Fracción activa')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(plots_dir / 'training_curves.png', dpi=300, bbox_inches='tight')
plt.show()

print('\n' + '='*50)
print('Gráficas guardadas exitosamente')
print('='*50)

### 11. Resumen Final

In [ ]:
print('\n' + '='*70)
print(' RESUMEN FINAL DEL ENTRENAMIENTO (TopK SAE)')
print('='*70)

arch_summary = pd.DataFrame({
    'Componente': ['Input (d_in)', 'Hidden (d_sae)', 'k (L0 fijo)', 'Expansión', 'Parámetros'],
    'Valor': [
        f'{cfg["d_in"]}',
        f'{cfg["d_sae"]}',
        f'{cfg["k"]} features activas',
        f'{cfg["d_sae"] // cfg["d_in"]}x',
        f'{num_params:,}'
    ]
})
print('\nArquitectura:')
print(arch_summary.to_string(index=False))

data_summary = pd.DataFrame({
    'Conjunto': ['Train', 'Validación', 'Total'],
    'Muestras': [f'{len(train_dataset):,}', f'{len(val_dataset):,}', f'{len(full_dataset):,}']
})
print('\nDatos:')
print(data_summary.to_string(index=False))

final_summary = pd.DataFrame({
    'Métrica': [
        'Val MSE best', 'Val MSE final',
        'Val L0 final (esperado k)', 'Val Fracción activa', 'Mejor época'
    ],
    'Valor': [
        f'{best_val_mse:.6f}',
        f"{history['val_mse_loss'][-1]:.6f}",
        f"{history['val_l0_sparsity'][-1]:.1f}  (k={cfg["k"]})",
        f"{history['val_fraction_active'][-1]:.4%}",
        f"{best_epoch}/{len(history['train_mse_loss'])}"
    ]
})
print('\nMétricas:')
print(final_summary.to_string(index=False))


### 12. Generar PDF con Quarto

Una vez ejecutado todo el notebook, ejecutar el siguiente comando en la terminal para generar el PDF con Quarto:

In [ ]:
import subprocess

exp_dir = get_experiment_dir(
    NAMING_META['experiment_base_path'],
    NAMING_META['variante'],
    NAMING_META['tecnica'],
    NAMING_META['capa'],
    NAMING_META['juegos'],
    extras_id=extras_id
)

pdf_name = get_report_name(
    NAMING_META['variante'],
    NAMING_META['tecnica'],
    NAMING_META['capa'],
    NAMING_META['juegos'],
    'training',
    extras_id=extras_id
)

notebook_name = 'train_sae_topk.ipynb'
output_path = exp_dir / pdf_name

print(f'Generando PDF con Quarto...')
print(f'Archivo de salida: {output_path}')

# Quarto no acepta rutas en --output, solo nombre de archivo
# Se usa --output-dir para el directorio y --output solo para el nombre
result = subprocess.run(
    f'quarto render {notebook_name} --to pdf --output-dir "{exp_dir}" --output {pdf_name}',
    shell=True,
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print(f'PDF generado exitosamente: {output_path}')
else:
    print(f'Error al generar PDF:')
    print(result.stderr)